In [1]:
import pandas as pd
import numpy as np
import math

df = pd.read_csv("demandsense_consistent.csv")
df["date"] = pd.to_datetime(df["date"])

print("Corrected dataset loaded")
print("Rows:", len(df))
print("Hotels:", df["hotel_id"].unique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

Corrected dataset loaded
Rows: 6905
Hotels: ['H001' 'H002' 'H003' 'H004' 'H005']
Date range: 2023-01-01 to 2026-10-12


In [2]:
hotel_id = "H001"

# Use the hotel's historical median room price as a starting estimate
room_price = float(
    df.loc[df["hotel_id"] == hotel_id, "avg_price"].median()
)

# Editable financial assumptions (₹)
room_service_cost = 700
direct_acquisition_cost = 200
direct_campaign_cost = 1500

# Channel assumptions
ota_commission_rate = 0.18
direct_discount_rate = 0.00
ota_discount_rate = 0.00

print("Hotel:", hotel_id)
print("Reference room price: ₹", round(room_price, 2))
print("Room service cost assumption: ₹", room_service_cost)
print("Direct acquisition cost: ₹", direct_acquisition_cost)
print("OTA commission assumption:", ota_commission_rate * 100, "%")

Hotel: H001
Reference room price: ₹ 4306.55
Room service cost assumption: ₹ 700
Direct acquisition cost: ₹ 200
OTA commission assumption: 18.0 %


In [3]:
channels = [
    {
        "Channel": "Direct",
        "Commission Rate": 0.00,
        "Discount Rate": direct_discount_rate,
        "Acquisition Cost": direct_acquisition_cost
    },
    {
        "Channel": "MakeMyTrip / OTA",
        "Commission Rate": ota_commission_rate,
        "Discount Rate": ota_discount_rate,
        "Acquisition Cost": 0
    }
]

results = []

for channel in channels:
    commission = room_price * channel["Commission Rate"]
    discount = room_price * channel["Discount Rate"]

    contribution = (
        room_price
        - commission
        - discount
        - channel["Acquisition Cost"]
        - room_service_cost
    )

    results.append({
        "Channel": channel["Channel"],
        "Room Price": round(room_price, 2),
        "Commission": round(commission, 2),
        "Discount": round(discount, 2),
        "Acquisition Cost": channel["Acquisition Cost"],
        "Service Cost": room_service_cost,
        "Contribution Per Booking": round(contribution, 2)
    })

profit_df = pd.DataFrame(results)
display(profit_df)

,Channel,Room Price,Commission,Discount,Acquisition Cost,Service Cost,Contribution Per Booking
0,Direct,4306.55,0.00,0.0,200,700,3406.55
1,MakeMyTrip / OTA,4306.55,775.18,0.0,0,700,2831.37


In [4]:
direct_contribution = float(
    profit_df.loc[
        profit_df["Channel"] == "Direct",
        "Contribution Per Booking"
    ].iloc[0]
)

ota_contribution = float(
    profit_df.loc[
        profit_df["Channel"] == "MakeMyTrip / OTA",
        "Contribution Per Booking"
    ].iloc[0]
)

def break_even_bookings(campaign_cost, contribution):
    if contribution <= 0:
        return None
    return math.ceil(campaign_cost / contribution)

break_even_df = pd.DataFrame([
    {
        "Strategy": "Direct Campaign",
        "Campaign Cost": direct_campaign_cost,
        "Contribution Per Booking": direct_contribution,
        "Additional Bookings Needed": break_even_bookings(
            direct_campaign_cost, direct_contribution
        )
    },
    {
        "Strategy": "OTA Promotion",
        "Campaign Cost": 0,
        "Contribution Per Booking": ota_contribution,
        "Additional Bookings Needed": 0
    }
])

display(break_even_df)

,Strategy,Campaign Cost,Contribution Per Booking,Additional Bookings Needed
0,Direct Campaign,1500,3406.55,1
1,OTA Promotion,0,2831.37,0


In [5]:
forecast_df = pd.read_csv("demandsense_30_day_forecast.csv")
forecast_df["date"] = pd.to_datetime(forecast_df["date"])

print("Forecast loaded:", len(forecast_df), "days")
display(forecast_df.head())

Forecast loaded: 30 days


,date,predicted_occupancy,total_rooms,predicted_occupied_rooms,predicted_vacant_rooms
0,2026-10-13,50.39,80,40,40
1,2026-10-14,50.18,80,40,40
2,2026-10-15,50.18,80,40,40
3,2026-10-16,50.18,80,40,40
4,2026-10-17,70.30,80,56,24


In [6]:

# Select a forecast date from the available 30-day forecast

forecast_df["date"] = pd.to_datetime(forecast_df["date"])

available_dates = sorted(
    forecast_df["date"].dt.strftime("%Y-%m-%d").unique()
)

print("Available forecast dates:")
for date in available_dates:
    print(date)

future_date = input("\nEnter forecast date (YYYY-MM-DD): ").strip()

if future_date not in available_dates:
    raise ValueError("Invalid date. Choose a date from the list above.")

selected_forecast = forecast_df[
    forecast_df["date"].dt.strftime("%Y-%m-%d") == future_date
].iloc[0]

predicted_occupancy = float(selected_forecast["predicted_occupancy"])
total_rooms = int(selected_forecast["total_rooms"])
expected_occupied_rooms = int(
    selected_forecast["predicted_occupied_rooms"]
)
expected_vacant_rooms = int(
    selected_forecast["predicted_vacant_rooms"]
)

estimated_room_revenue = expected_occupied_rooms * room_price

print("\nDEMANDSENSE FORECAST SUMMARY")
print("Hotel:", hotel_id)
print("Forecast date:", future_date)
print("Predicted occupancy:", round(predicted_occupancy, 2), "%")
print("Total rooms:", total_rooms)
print("Expected occupied rooms:", expected_occupied_rooms)
print("Expected vacant rooms:", expected_vacant_rooms)
print("Estimated room revenue: ₹", round(estimated_room_revenue, 2))


Available forecast dates:
2026-10-13
2026-10-14
2026-10-15
2026-10-16
2026-10-17
2026-10-18
2026-10-19
2026-10-20
2026-10-21
2026-10-22
2026-10-23
2026-10-24
2026-10-25
2026-10-26
2026-10-27
2026-10-28
2026-10-29
2026-10-30
2026-10-31
2026-11-01
2026-11-02
2026-11-03
2026-11-04
2026-11-05
2026-11-06
2026-11-07
2026-11-08
2026-11-09
2026-11-10
2026-11-11



Enter forecast date (YYYY-MM-DD):  2026-10-20



DEMANDSENSE FORECAST SUMMARY
Hotel: H001
Forecast date: 2026-10-20
Predicted occupancy: 50.78 %
Total rooms: 80
Expected occupied rooms: 41
Expected vacant rooms: 39
Estimated room revenue: ₹ 176568.55


In [7]:

# Campaign comparison
# Extra bookings are assumptions, not AI predictions.

expected_extra_direct_bookings = None
expected_extra_ota_bookings = None

direct_campaign_cost = 1500
ota_campaign_cost = 0  # Placeholder: enter the actual promotion cost

direct_contribution = (
    room_price - direct_acquisition_cost - room_service_cost
)

ota_contribution = (
    room_price * (1 - ota_commission_rate) - room_service_cost
)

print("CAMPAIGN SCENARIO COMPARISON")
print("----------------------------")
print("Direct contribution per extra booking: ₹", round(direct_contribution, 2))
print("OTA contribution per extra booking: ₹", round(ota_contribution, 2))

if expected_extra_direct_bookings is None or expected_extra_ota_bookings is None:
    print("\nNo campaign winner recommended.")
    print("Enter defensible estimates for extra bookings in both variables.")
    print("Use hotel campaign history or clearly labelled hypothetical scenarios.")
else:
    direct_bookings = max(0, int(expected_extra_direct_bookings))
    ota_bookings = max(0, int(expected_extra_ota_bookings))

    direct_bookings = min(direct_bookings, expected_vacant_rooms)
    ota_bookings = min(ota_bookings, expected_vacant_rooms)

    direct_gain = (
        direct_bookings * direct_contribution - direct_campaign_cost
    )
    ota_gain = (
        ota_bookings * ota_contribution - ota_campaign_cost
    )

    print("\nDirect campaign incremental contribution: ₹", round(direct_gain, 2))
    print("OTA promotion incremental contribution: ₹", round(ota_gain, 2))

    if direct_gain <= 0 and ota_gain <= 0:
        print("Recommendation: Do not run either campaign under these assumptions.")
    elif direct_gain > ota_gain:
        print("Recommendation: Direct campaign has the higher estimated contribution.")
    elif ota_gain > direct_gain:
        print("Recommendation: OTA promotion has the higher estimated contribution.")
    else:
        print("Recommendation: Both campaigns have equal estimated contribution.")

    print("Important: These results depend on the booking-uplift assumptions.")


CAMPAIGN SCENARIO COMPARISON
----------------------------
Direct contribution per extra booking: ₹ 3406.55
OTA contribution per extra booking: ₹ 2831.37

No campaign winner recommended.
Enter defensible estimates for extra bookings in both variables.
Use hotel campaign history or clearly labelled hypothetical scenarios.


In [9]:

print("DEMANDSENSE PROFIT ENGINE SUMMARY")
print("=================================")
print("Hotel:", hotel_id)
print("Forecast date:", future_date)
print("Forecast occupancy:", round(predicted_occupancy, 2), "%")
print("Expected occupied rooms:", expected_occupied_rooms)
print("Expected vacant rooms:", expected_vacant_rooms)
print("Estimated room revenue: ₹", round(estimated_room_revenue, 2))

print("\nContribution per booking:")
print("Direct: ₹", round(direct_contribution, 2))
print("OTA: ₹", round(ota_contribution, 2))

print("\nCampaign decision:")
if (
    expected_extra_direct_bookings is None
    or expected_extra_ota_bookings is None
):
    print("Pending: campaign booking-uplift evidence is required.")
else:
    print("See the campaign comparison cell for the estimated results.")

print("\nNote: Forecasts use synthetic demonstration data.")


DEMANDSENSE PROFIT ENGINE SUMMARY
Hotel: H001
Forecast date: 2026-10-20
Forecast occupancy: 50.78 %
Expected occupied rooms: 41
Expected vacant rooms: 39
Estimated room revenue: ₹ 176568.55

Contribution per booking:
Direct: ₹ 3406.55
OTA: ₹ 2831.37

Campaign decision:
Pending: campaign booking-uplift evidence is required.

Note: Forecasts use synthetic demonstration data.


In [10]:

# Final DemandSense summary

print("=" * 45)
print("       DEMANDSENSE PROFIT ENGINE")
print("=" * 45)

print("Hotel:", hotel_id)
print("Forecast date:", future_date)
print("Predicted occupancy:", round(predicted_occupancy, 2), "%")
print("Occupied rooms:", expected_occupied_rooms, "/", total_rooms)
print("Vacant rooms:", expected_vacant_rooms)
print("Estimated room revenue: ₹", round(estimated_room_revenue, 2))

print("\nCHANNEL ECONOMICS")
print("-" * 45)
print("Direct contribution per booking: ₹", round(direct_contribution, 2))
print("OTA contribution per booking: ₹", round(ota_contribution, 2))

print("\nDECISION STATUS")
print("-" * 45)

if (
    expected_extra_direct_bookings is None
    or expected_extra_ota_bookings is None
):
    print("Campaign recommendation: Pending booking-uplift evidence.")
    print("Reason: Additional campaign bookings are not yet established.")
else:
    print("See the campaign comparison above.")
    print("Results depend on the booking-uplift assumptions.")

print("\nDATA LIMITATION")
print("Forecasts use synthetic demonstration data.")
print("Revenue and campaign outcomes are estimates, not guarantees.")
print("=" * 45)


       DEMANDSENSE PROFIT ENGINE
Hotel: H001
Forecast date: 2026-10-20
Predicted occupancy: 50.78 %
Occupied rooms: 41 / 80
Vacant rooms: 39
Estimated room revenue: ₹ 176568.55

CHANNEL ECONOMICS
---------------------------------------------
Direct contribution per booking: ₹ 3406.55
OTA contribution per booking: ₹ 2831.37

DECISION STATUS
---------------------------------------------
Campaign recommendation: Pending booking-uplift evidence.
Reason: Additional campaign bookings are not yet established.

DATA LIMITATION
Forecasts use synthetic demonstration data.
Revenue and campaign outcomes are estimates, not guarantees.


In [ ]:

%pip install streamlit pandas matplotlib
